# ReAct Research Agent with Tavily + Firecrawl

Build a research agent that **searches the web** (Tavily) and **reads full pages** (Firecrawl),
reasoning step-by-step with lionagi's ReAct loop.

**Setup**: Add to your `.env`:
```
TAVILY_API_KEY=tvly-...
FIRECRAWL_API_KEY=fc-...
OPENAI_API_KEY=sk-...
```

## 1. Define Search & Scrape Tools

We wrap Tavily search and Firecrawl scrape as plain async functions.
lionagi auto-generates tool schemas from the signatures.

In [ ]:
from lionagi.service.connections.match_endpoint import match_endpoint

_tavily = match_endpoint("tavily", "search")
_firecrawl = match_endpoint("firecrawl", "scrape")


async def web_search(
    query: str,
    max_results: int = 5,
    topic: str = "general",
) -> str:
    """Search the web using Tavily. Returns titles, URLs, and snippets.

    Args:
        query: What to search for.
        max_results: Number of results (1-20, default 5).
        topic: Search category - 'general', 'news', or 'finance'.
    """
    resp = await _tavily.call(
        {"query": query, "max_results": max_results, "topic": topic, "include_answer": True}
    )
    parts = []
    if resp.get("answer"):
        parts.append(f"Quick answer: {resp['answer']}\n")
    for r in resp.get("results", []):
        parts.append(f"- [{r['title']}]({r['url']})\n  {r.get('content', '')[:300]}")
    return "\n".join(parts) or "No results found."


async def read_page(url: str) -> str:
    """Scrape and read the full content of a web page. Returns markdown text.

    Args:
        url: The URL to read.
    """
    resp = await _firecrawl.call({"url": url, "onlyMainContent": True})
    if resp.get("success") and resp.get("data"):
        md = resp["data"].get("markdown", "")
        return md[:5000] if md else "Page loaded but no text content extracted."
    return f"Failed to read page: {resp}"


print("Tools ready: web_search, read_page")

## 2. Quick Tool Test

Verify both tools work before handing them to the agent.

In [ ]:
results = await web_search("latest developments in AI agents 2025", max_results=3)
print(results)

In [ ]:
page = await read_page("https://github.com/lion-agi/lionagi")
print(page[:1000])

## 3. ReAct Research Agent

The agent can **search → read → reason → search again** in a loop.
It decides on its own when to search, when to read a full page, and when it has enough info.

In [ ]:
from lionagi import Branch, iModel

researcher = Branch(
    system=(
        "You are a research assistant with web search and page reading capabilities. "
        "When answering questions:\n"
        "1. Search the web to find relevant sources\n"
        "2. Read promising pages for detailed information\n"
        "3. Synthesize findings into a clear, sourced answer\n"
        "Always cite your sources with URLs."
    ),
    tools=[web_search, read_page],
    chat_model=iModel(provider="openai", model="gpt-4.1-mini"),
)

print("Research agent ready.")

In [ ]:
answer = await researcher.ReAct(
    instruct={
        "instruction": (
            "What are the key differences between Tavily and Perplexity APIs "
            "for building AI agent search tools? Compare pricing, features, "
            "and best use cases. Read at least one page for detailed info."
        ),
    },
    max_extensions=8,
    verbose=True,
)

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(answer)

## 4. Structured Output with ReAct

Get research results as a Pydantic model instead of free text.

In [ ]:
from lionagi import BaseModel, Field


class Source(BaseModel):
    title: str
    url: str


class ResearchReport(BaseModel):
    title: str = Field(description="Report title")
    summary: str = Field(description="2-3 sentence executive summary")
    key_findings: list[str] = Field(description="Main findings as bullet points")
    sources: list[Source] = Field(description="Sources used")


agent2 = Branch(
    system=(
        "You are a research analyst. Search the web, read pages for detail, "
        "and produce structured research reports with citations."
    ),
    tools=[web_search, read_page],
    chat_model=iModel(provider="openai", model="gpt-4.1-mini"),
)

report = await agent2.ReAct(
    instruct={
        "instruction": (
            "Research the current state of open-source AI agent frameworks in 2025. "
            "Which are the most popular? What are the key trends?"
        ),
    },
    max_extensions=8,
    verbose=True,
    response_format=ResearchReport,
)

print("\n" + "=" * 60)
print(f"Title: {report.title}")
print(f"\nSummary: {report.summary}")
print("\nKey Findings:")
for f in report.key_findings:
    print(f"  - {f}")
print("\nSources:")
for s in report.sources:
    print(f"  - [{s.title}]({s.url})")

## 5. Inspect the Conversation

See every step the agent took — reasoning, tool calls, and observations.

In [ ]:
df = agent2.to_df()
print(f"Total conversation steps: {len(df)}")
print()

for _, row in df.iterrows():
    role = row["role"]
    content_str = str(row["content"])[:120]
    if "user" in str(role):
        print(f"  USER: {content_str}...")
    elif "assistant" in str(role):
        print(f"  ASSISTANT: {content_str}...")
    elif "action" in str(role):
        print(f"  TOOL: {content_str}...")
    elif "system" in str(role):
        print(f"  SYSTEM: (system message)")

## 6. Using Tavily & Firecrawl Directly (Without ReAct)

You can also use the endpoints directly for programmatic access.

In [ ]:
from lionagi.service.connections.match_endpoint import match_endpoint

# Tavily search
tavily = match_endpoint("tavily", "search")
results = await tavily.call({"query": "lionagi framework", "max_results": 3, "include_answer": True})
print("Tavily answer:", results.get("answer"))
print(f"Got {len(results['results'])} results\n")

# Firecrawl scrape
firecrawl = match_endpoint("firecrawl", "scrape")
page = await firecrawl.call({"url": results["results"][0]["url"], "onlyMainContent": True})
if page.get("success"):
    print("Scraped page preview:")
    print(page["data"]["markdown"][:500])

# Firecrawl map
mapper = match_endpoint("firecrawl", "map")
sitemap = await mapper.call({"url": "https://github.com/lion-agi/lionagi", "limit": 10})
print("\nMapped URLs:")
for link in sitemap.get("links", [])[:5]:
    print(f"  {link}")